# Combining the Spectra of the 3 EPIC Cameras -- Part 2: Applying Models to the Spectra
## Shortened Version
<hr style="border: 2px solid #f5bf03" />

- **Description:** Applying various XSPEC models to the spectra using pyXSPEC and evaluating the fitness of the models.
- **Level:** Intermediate
- **Data:** XMM observation of the Circinus Galaxy (obsid=0111240101)
- **Requirements:** Must be run using pySAS version 2.2.7 or higher.
- **Credit:** Ryan Tanner (October 2025), based on an <a href="https://www.cosmos.esa.int/web/xmm-newton/sas-thread-epic-merging">ESA SOC SAS Tutorial</a>
- **Support:** <a href="https://heasarc.gsfc.nasa.gov/cgi-bin/Feedback">XMM Newton GOF Helpdesk</a>
- **Last verified to run:** 30 June 2026, for SAS v22.1 and pySAS v2.5.0

<hr style="border: 2px solid #f5bf03" />

## 1. Introduction

This is Part 2. By the end of Part 1 you should have files of a source spectrum, a background spectrum, a RMF, and a ARF for all three EPIC cameras.

<div class="alert alert-block alert-info">
    <b>Note:</b> This is a shortened version of the main notebook. This only contains the final plot as shown in Molendi et al. (2003). It does not explain the process to get there.
</div>


Reference: Molendi S., Bianchi S., Matt G., 2003, MNRAS, 343, L1 [10.1046/j.1365-8711.2003.06783.x](https://doi.org/10.1046/j.1365-8711.2003.06783.x)

#### SAS Tasks to be Used

**None**

#### Useful Links

- [`pysas` Documentation](https://github.com/XMMGOF/pysas_docs "pysas Documentation")
- [`pysas` on GitHub](https://github.com/XMMGOF/pysas)
- [Common SAS Threads](https://www.cosmos.esa.int/web/xmm-newton/sas-threads/ "SAS Threads")
- [Users' Guide to the XMM-Newton Science Analysis System (SAS)](https://xmm-tools.cosmos.esa.int/external/xmm_user_support/documentation/sas_usg/USG/SASUSG.html "Users' Guide")
- [The XMM-Newton ABC Guide](https://heasarc.gsfc.nasa.gov/docs/xmm/abc/ "ABC Guide")
- [XMM Newton GOF Helpdesk](https://heasarc.gsfc.nasa.gov/cgi-bin/Feedback "Helpdesk") - Link to form to contact the XMM-Newton GOF Helpdesk.

<div class="alert alert-block alert-warning">
    <b>Warning:</b> By default this notebook will place observation data files in your default <tt>data_dir</tt> directory. Make sure pySAS has been configured properly.
</div>

## 2. Link to Previously Generated Files

In [ ]:
# pySAS imports
import pysas
from pysas import MyTask

# Importing PyXSPEC
import xspec

# Useful imports
import os, glob, shutil

# Imports for plotting
import matplotlib.pyplot as plt
from astropy.visualization import astropy_mpl_style
from astropy.io import fits
from astropy.wcs import WCS
from astropy.table import Table
from regions import CircleSkyRegion
from astropy.coordinates import SkyCoord
from astropy.visualization import ZScaleInterval, ImageNormalize
import astropy.units as u
from matplotlib.ticker import StrMethodFormatter
from IPython.display import Image, display
plt.style.use(astropy_mpl_style)

# To handle certain warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
obsid = '0111240101'
my_obs = pysas.ObsID(obsid)
instruments = ['EMOS1','EMOS2','EPN']

clean_event_lists   = {}
hi_res_images       = {}
source_event_list   = {}
bkg_event_list      = {}
source_spectra_file = {}
bkg_spectra_file    = {}
rmf_file            = {}
arf_file            = {}
grouped_spectra     = {}

for instrument in instruments:
    clean_event_lists[instrument]   = f'{instrument}_clean_event_list.fits'
    hi_res_images[instrument]       = f'{instrument}_image.fits'
    source_event_list[instrument]   = f'{instrument}_source_event_list.fits'
    bkg_event_list[instrument]      = f'{instrument}_bkg_event_list.fits'
    source_spectra_file[instrument] = f'{instrument}_pi.fits'
    bkg_spectra_file[instrument]    = f'{instrument}_bkg_pi.fits'
    rmf_file[instrument]            = f'{instrument}_rmf.fits'
    arf_file[instrument]            = f'{instrument}_arf.fits'
    grouped_spectra[instrument]     = f'{instrument}_grp.fits'

filepha="src_spectrum_grp.ds"
filebkg="bkg_spectrum_grp.ds"
filersp="response_grp.rmf"

In [ ]:
def plot_spectrum(spectrum,plot_file_name='spectrum_plot.png'):
    xspec.Plot.device='/null'
    xspec.Plot.xAxis = 'keV'

    # Pull off data for main plot
    xspec.Plot('data')
    energy = xspec.Plot.x()
    counts = xspec.Plot.y()
    xErrs = xspec.Plot.xErr()
    yErrs = xspec.Plot.yErr()

    # Get bin edges for "stairs" plot
    bin_edges = []
    for i in spectrum.energies: bin_edges.append(i[0])
    bin_edges.append(spectrum.energies[-1][1])

    # Make the figure and two subplots
    fig, ax0 = plt.subplots(figsize=(9, 7))

    # Main plot
    ax0.errorbar(energy, counts, yerr=yErrs, xerr=xErrs, linestyle='', marker='')
    #ax0.set_xscale('log')
    ax0.set_yscale('log')
    ax0.set_xlim([bin_edges[0], bin_edges[-1]])
    ax0.tick_params(top=True,axis="x",direction="in",which='both')
    ax0.tick_params(axis="y",direction="in",which='both',right=True)
    ax0.set_ylabel('counts sec$^{-1}$ keV$^{-1}$')
    ax0.set_title('Data')

    # Save plot to file
    fig.savefig(plot_file_name)

    #return fig, ax0

In [ ]:
def plot_data_model(spectrum,plot_file_name='data_model_plot.png',rebin=None):
    xspec.Plot.device='/null'
    xspec.Plot.xAxis = 'keV'
    if rebin is None:
        xspec.Plot.setRebin(minSig=1, maxBins=1)
    else:
        xspec.Plot.setRebin(minSig=rebin[0], maxBins=rebin[1])

    # Pull off data for main plot
    xspec.Plot('data')
    energy = xspec.Plot.x()
    counts = xspec.Plot.y()
    folded = xspec.Plot.model()
    xErrs = xspec.Plot.xErr()
    yErrs = xspec.Plot.yErr()

    # Pull off data for ratio plot
    xspec.Plot('ratio')
    ratio = xspec.Plot.y()
    r_xerror = xspec.Plot.xErr()
    r_yerror = xspec.Plot.yErr()

    # Get bin edges for "stairs" plot
    if rebin is None:
        bin_edges = []
        for i in spectrum.energies: bin_edges.append(i[0])
        bin_edges.append(spectrum.energies[-1][1])
    else:
        bin_edges = list(energy)
        bin_edges.append(energy[-1]+(energy[-1]-energy[-2]))

    # Make the figure and two subplots
    fig, (ax0, ax1) = plt.subplots(nrows=2, sharex=True, height_ratios=[2.5, 1],figsize=(9, 7))

    # Main plot
    ax0.errorbar(energy, counts, yerr=yErrs, xerr=xErrs, linestyle='', marker='', zorder=1)
    ax0.stairs(folded,bin_edges, color='r', zorder=2)
    ax0.set_xscale('log')
    ax0.set_yscale('log')
    ax0.set_xlim([bin_edges[0], bin_edges[-1]])
    ax0.tick_params(top=True,axis="x",direction="in",which='both')
    ax0.tick_params(axis="y",direction="in",which='both',right=True)
    ax0.set_ylabel('counts sec$^{-1}$ keV$^{-1}$')
    ax0.set_title('Data and Folded Model')
    ax0.grid(which='minor')

    # Ratio plot
    ax1.errorbar(energy, ratio, yerr=r_yerror, xerr=r_xerror, linestyle='', marker='')
    ax1.axhline(y=1, color='g')
    ax1.set_xscale('log')
    ax1.tick_params(top=True,axis="x",direction="in",which='both')
    ax1.tick_params(axis="y",direction="in",which='both')
    ax1.xaxis.set_major_formatter(StrMethodFormatter('{x:.1f}'))
    ax1.xaxis.set_minor_formatter(StrMethodFormatter('{x:.1f}'))
    ax1.set_xlabel('Energy (keV)')
    ax1.set_ylabel('Ratio')
    ax1.grid(which='minor')

    # This puts the plots together with no space in between
    plt.subplots_adjust(hspace=.0)

    # Save plot to file
    fig.savefig(plot_file_name)

    return fig, ax0, ax1

## 3. Load Spectra into XSPEC

In [ ]:
xspec.AllData.clear()

In [ ]:
xspec.AllData.clear()
s = [xspec.Spectrum(grouped_spectra['EPN']),
     xspec.Spectrum(grouped_spectra['EMOS1']),
     xspec.Spectrum(grouped_spectra['EMOS2']),
     xspec.Spectrum(filepha)]
for spectrum in s:
    spectrum.background = ""
    spectrum.ignore('0.0-0.5,13.0-**')

In [ ]:
# Make the figure
fig, ax0 = plt.subplots(figsize=(9, 7))

colors = ['#02c14d','#0804f9','#f7022a','#aa23ff']
labels = ['EPN', 'EMOS1', 'EMOS2', 'Merged EPIC']

xspec.Plot.device='/null'
xspec.Plot.xAxis = 'keV'

for i,spectrum in enumerate(s):
    # Pull off data for main plot
    xspec.Plot('data')
    energy = xspec.Plot.x(i+1,1)
    counts = xspec.Plot.y(i+1,1)
    xErrs = xspec.Plot.xErr(i+1,1)
    yErrs = xspec.Plot.yErr(i+1,1)
    
    # Main plot
    ax0.errorbar(energy, counts, yerr=yErrs, xerr=xErrs, linestyle='', marker='', color=colors[i], label=labels[i])
    ax0.set_xscale('log')
    #ax0.set_yscale('log')
    #ax0.set_xlim(0.5, 13.0)
    #ax0.set_ylim(0.0, 1.0)
    ax0.tick_params(top=True,axis="x",direction="in",which='both')
    ax0.tick_params(axis="y",direction="in",which='both',right=True)
    ax0.set_ylabel('counts sec$^{-1}$ keV$^{-1}$')
    ax0.set_title(f'Circinus Galaxy - Obs ID: {my_obs.obsid}')
    ax0.grid(which='minor')
    ax0.legend(loc='upper left')
    ax0.grid(which='minor')

# Save plot to file
fig.savefig('EPIC_spectra.png')

In the plot above we have the spectra from each of the EPIC cameras along with the merged spectrum.

## 5. More Comlicated Models
### 5.2 PEXRAV Model

Following Molendi et al. (2003) we will consider a [`pexrav` model](https://heasarc.gsfc.nasa.gov/docs/software/xspec/manual/node215.html) which consists of an exponentially cut off power law spectrum reflected from neutral material. We will add to this three gaussians for iron and nickel emission lines.

We will set the photon index for the `pexrav` model to 1.56, the high energy cut off to 56 keV, and the redshift to 0.0015, and freeze all these parameters in place. And we will unfreeze the abundance.

To start out we will consider an absoption model along with the `pexrav` model. Then we will add in the three emission lines, and finally we will fit the model over the range of 4.5-11 keV to reproduce Figure 2 from Molendi et al. (2003). (Note: The error bars in our spectrum will be slightly different since we deal with the background differently.)

Reference: Molendi S., Bianchi S., Matt G., 2003, MNRAS, 343, L1 [10.1046/j.1365-8711.2003.06783.x](https://doi.org/10.1046/j.1365-8711.2003.06783.x)

In [ ]:
xspec.AllData.clear()
xspec.AllModels.clear()
spectrum = xspec.Spectrum(filepha)
spectrum.background = ""
spectrum.ignore('0.0-0.5,11.0-**')

In [ ]:
# Model #2
xspec.AllModels.clear()
mod = xspec.Model('phabs *(pexrav + ga + ga + ga)')
set_pars = {2:1.56, 3:56.0, 5:0.0015, 10:6.4, 13:7.05, 16:7.4}
mod.setPars(set_pars)
par = mod(2)
par.frozen = True
par = mod(3)
par.frozen = True
par = mod(6)
par.frozen = False
# The next two lines restrict the range for the third gaussian so that the algorithm finds the "correct" value
# The values for the parameter are:
# "parameter value, delta (not set), hard lower value, soft lower value, soft upper value, hard upper value"
par = mod(16)
par.values = "7.4,,7.1,7.2,7.6,7.7"
xspec.Fit.renorm()
xspec.Fit.perform()

In [ ]:
plot_data_model(spectrum)

In [ ]:
# Model #3
xspec.AllData.clear()
spectrum = xspec.Spectrum(filepha)
spectrum.background = ""
spectrum.ignore('0.0-4.5,11.0-**')

xspec.AllModels.clear()
mod = xspec.Model('pexrav + ga + ga + ga')
set_pars = {1:1.56, 2:56.0, 4:0.0015, 9:6.4, 12:7.05, 15:7.4}
mod.setPars(set_pars)
par = mod(1)
par.frozen = True
par = mod(2)
par.frozen = True
# The next two lines restrict the range for the third gaussian so that the algorithm finds the "correct" value
# The values for the parameter are:
# "parameter value, delta (not set), hard lower value, soft lower value, soft upper value, hard upper value"
par = mod(15)
par.values = "7.4,,7.1,7.2,7.6,7.7"
xspec.Fit.renorm()
xspec.Fit.perform()

Compare the plot from the following cell with Figure 2 (below) from Molendi et al. (2003).

In [ ]:
plot_data_model(spectrum,rebin=(8,60))

![Pileup Plot](./_files/Molendi_Fig_2.jpeg)